In [ ]:
# ✅ ConversationBufferMemory — stores ALL messages manually 
# Modern LangChain (LCEL) manages memory by passing chat_history directly 
 
from langchain_core.chat_history import InMemoryChatMessageHistory 
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory 
from langchain_core.output_parsers import StrOutputParser 
from langchain_core.messages import HumanMessage, AIMessage 
from dotenv import load_dotenv
#from langchain.prompts import ChatPromptTemplate

from langchain_anthropic import ChatAnthropic
import os
 
load_dotenv(override=True)
 
api_key = os.getenv("ANTHROPIC_API_KEY")
if api_key is None:
    raise ValueError("ANTHROPIC_API_KEY not found in .env file")
base_url=os.getenv("BASE_URL")
model_name=os.getenv("MODEL_NAME")
 
llm = ChatAnthropic(model=model_name,base_url=base_url,anthropic_api_key=api_key, max_tokens=200)

# Prompt with a placeholder for history 
prompt = ChatPromptTemplate.from_messages([ 
    ('system', 'You are a helpful assistant. Be concise.'), 
    MessagesPlaceholder(variable_name='chat_history'),   # ← history injected here  # type: ignore
    ('human', '{input}'), 
]) 
 
chain = prompt | llm | StrOutputParser() 
 
# In-memory store — one history object per session_id 
store = {} 
 
def get_session_history(session_id: str): 
    if session_id not in store: 
        store[session_id] = InMemoryChatMessageHistory() 
    return store[session_id] 
 
# Wrap chain with message history (Buffer — keeps ALL messages) 
chain_with_memory = RunnableWithMessageHistory( 
    chain, 
    get_session_history, 
    input_messages_key='input', 
    history_messages_key='chat_history', 
) 
 
config = {'configurable': {'session_id': 'session_1'}} 
 
# Turn 1 
r1 = chain_with_memory.invoke({'input': 'My name is Ali and I love Python.'}, config=config) 
print('Turn 1:', r1) 
print(config) 
 
# Turn 2 
r2 = chain_with_memory.invoke({'input': 'What is my name?'}, config=config) 
print('Turn 2:', r2)   # Now correctly answers: Ali! 
print(config) 
 
# Turn 3 
r3 = chain_with_memory.invoke({'input': 'What programming language do I love?'}, config=config) 
print('Turn 3:', r3)   # Correctly answers: Python! 
 
# Inspect full stored history 
print('\n--- Full History ---') 
for msg in store['session_1'].messages: 
    print(f'{msg.__class__.__name__}: {msg.content}') 


/Software/langenv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3701: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


Turn 1: Hi Ali! That's great — Python is an awesome language! 🐍 Is there something Python-related I can help you with?
{'configurable': {'session_id': 'session_1'}}
Turn 2: Your name is Ali! 😊
{'configurable': {'session_id': 'session_1'}}
Turn 3: You love Python! 🐍

--- Full History ---
HumanMessage: My name is Ali and I love Python.
AIMessage: Hi Ali! That's great — Python is an awesome language! 🐍 Is there something Python-related I can help you with?
HumanMessage: What is my name?
AIMessage: Your name is Ali! 😊
HumanMessage: What programming language do I love?
AIMessage: You love Python! 🐍
